# 03 · Pilot: temperature sweep and failure prediction (design doc §8.8)

**How to run**: pick a Colab server with a GPU (a T4 is enough) and click "Run All"; about 2.5 hours. If the connection drops, click "Run All" again; finished parts are skipped. Results go to `DATA_DIR/03_pilot_temperature/`.

**What it computes**: Qwen2.5-3B continues 100 texts at five temperatures (0.6, 0.8, 0.9, 1.1, 1.2; top-p fixed at 1.0); with the T = 1.0 texts of notebook 01 this gives six temperatures. Three scorers (GPT-2, GPT-2 XL and Qwen2.5-3B itself) score them.

Before looking at AUROC, the notebook computes the margin of every (temperature, scorer) pair, mean human surprisal minus mean AI surprisal, and records the predicted failure temperature (where the margin is 0). Only then does it show AUROC for comparison.

In [ ]:
# 1. Check for a GPU.
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to a GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")

In [ ]:
# 2. Mount Google Drive and switch to the data folder.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
os.makedirs("03_pilot_temperature", exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"

In [ ]:
# 3. Install packages and load the human texts of notebook 01.
%pip install -q -U transformers huggingface-hub accelerate
import json, gc, glob, shutil, traceback, time
import numpy as np, pandas as pd
humans = json.load(open("01_pilot_decoding/humans.json"))
print(len(humans), "human texts")

In [ ]:
# 4. Continuations: Qwen2.5-3B at five temperatures, top-p fixed at 1.0. T = 1.0 reuses the 01_pilot_decoding file.
import token_info
GEN = "Qwen/Qwen2.5-3B"
TEMPS = [0.6, 0.8, 0.9, 1.1, 1.2]
PREFIX = 30
shutil.copy("01_pilot_decoding/gen_Qwen2.5-3B_pure.json", "03_pilot_temperature/gen_T1.0.json") if not os.path.exists("03_pilot_temperature/gen_T1.0.json") else None

for temp in TEMPS:
    path = f"03_pilot_temperature/gen_T{temp}.json"
    if os.path.exists(path):
        print("Already done, skipping:", path); continue
    engine = None
    try:
        token_info.SETTINGS.update(scoring_model=GEN, sampling_model=GEN, temperature=temp, top_p=1.0, do_sample=True,
                                   prefix_tokens=PREFIX, paraphrase_enabled=False, save_token_features=False,
                                   cache_dir="/content/hf", seed=42)
        engine = token_info.TokenInfo()
        rows, start = [], time.time()
        for h in humans:
            try:
                r = engine.continue_text(h["text"])
            except (ValueError, RuntimeError) as e:
                print("Skipping text", h["id"], ":", e); continue
            ids = engine.encode(h["text"])
            rows.append(dict(id=h["id"], domain=h["domain"], model=GEN, decoding=f"T{temp}", prefix=r["prefix"],
                             ai=r["continuation"],
                             human=engine.scoring_tokenizer.decode(ids[PREFIX:], clean_up_tokenization_spaces=False)))
            if len(rows) % 20 == 0:
                print(f"  T={temp}: {len(rows)}/{len(humans)} texts, {(time.time() - start) / 60:.0f} min so far", flush=True)
        json.dump(rows, open(path, "w"), ensure_ascii=False)
        print("T =", temp, "done:", len(rows), "texts")
    except Exception:
        traceback.print_exc(); print("This temperature failed, skipping:", temp)
    finally:
        engine = None; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 5. Scoring: each of the three scorers scores the human and AI texts at all six temperatures.
from transformers import AutoTokenizer, AutoModelForCausalLM
SCORERS = ["gpt2", "gpt2-xl", "Qwen/Qwen2.5-3B"]

def score_texts(scorer_name, records):
    tok = AutoTokenizer.from_pretrained(scorer_name)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
    model = AutoModelForCausalLM.from_pretrained(scorer_name, dtype=dtype).cuda().eval()
    out = []
    with torch.no_grad():
        for i, r in enumerate(records):
            k = len(tok(r["prefix"])["input_ids"])
            for who in ("human", "ai"):
                ids = tok(r["prefix"] + r[who], truncation=True, max_length=512)["input_ids"]
                if len(ids) - k < 10:
                    continue
                x = torch.tensor([ids]).cuda()
                lp = torch.log_softmax(model(x).logits[0, :-1].float(), -1)
                p = lp.exp(); e1 = (p * lp).sum(-1)
                rows = [-lp.gather(1, x[0, 1:, None])[:, 0], -e1, (p * lp * lp).sum(-1) - e1 ** 2,
                        p.topk(10, -1).values.sum(-1)]
                out.append(dict(id=r["id"], domain=r["domain"], decoding=r["decoding"], scorer=scorer_name, who=who,
                                arr=torch.stack(rows).cpu().numpy().astype(np.float32)[:, k - 1:]))
            if i % 100 == 0:
                print(" ", scorer_name, i, "/", len(records), flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

recs = []
for f in sorted(glob.glob("03_pilot_temperature/gen_T*.json")):
    recs += json.load(open(f))
for sc in SCORERS:
    path = f"03_pilot_temperature/scores_{sc.split('/')[-1]}.pkl"
    if os.path.exists(path):
        print("Already done, skipping:", path); continue
    try:
        pd.DataFrame(score_texts(sc, recs)).to_pickle(path)
        print(sc, "done")
    except Exception:
        traceback.print_exc(); print("This scorer failed, skipping:", sc)
    finally:
        gc.collect(); torch.cuda.empty_cache()
        shutil.rmtree(f"/content/hf/models--{sc.replace('/', '--')}", ignore_errors=True)

In [ ]:
# 6. First the margin and the predicted failure temperature (surprisal means only), then AUROC.
from sklearn.metrics import roc_auc_score
s = pd.concat([pd.read_pickle(f) for f in sorted(glob.glob("03_pilot_temperature/scores_*.pkl"))], ignore_index=True)
s["surprisal"] = [a[0].mean() for a in s.arr]
s["fdg"] = [(a[1].sum() - a[0].sum()) / np.sqrt(a[2].sum()) for a in s.arr]
s["y"] = (s.who == "ai").astype(int)
s["T"] = s.decoding.replace("pure", "T1.0").str[1:].astype(float)  # files reused from 01_pilot_decoding have decoding "pure"

def per(g):
    h, ai = g[g.y == 0], g[g.y == 1]
    return pd.Series({"surprisal_human": h.surprisal.mean(), "surprisal_ai": ai.surprisal.mean(),
                      "surprisal_margin": h.surprisal.mean() - ai.surprisal.mean(), "AUROC": roc_auc_score(g.y, g.fdg)})
table = s.groupby(["scorer", "T"]).apply(per)
print("== Margin (look at this first) ==")
print(table[["surprisal_human", "surprisal_ai", "surprisal_margin"]].round(3))
for sc, g in table.reset_index().groupby("scorer"):
    g = g.sort_values("T")
    cross = np.interp(0, -g.surprisal_margin.values, g["T"].values) if (g.surprisal_margin.min() < 0 < g.surprisal_margin.max()) else None
    print(f"{sc}: predicted failure temperature {cross:.2f}" if cross else f"{sc}: the margin does not change sign in this range")
print("\n== AUROC (look at this second) ==")
print(table[["AUROC"]].round(3))
table.to_csv("03_pilot_temperature/summary.csv")

In [ ]:
# 7. Flush unsynced files back to Google Drive.
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/03_pilot_temperature/")